# Trees chapter project - the Titanic running score

Build one classifier **three ways** on the same Titanic split and keep a **running scoreboard**:
a single tree (HW1) -> a random forest (HW2) -> gradient boosting (HW3). Report accuracy + F1 / ROC-AUC
each time and watch how - and *whether* - each ensemble beats the last.

Data is pinned in `data/titanic.csv` (from OpenML), so nothing is fetched at run time.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

SEED = 509

df = pd.read_csv('data/titanic.csv')
df['sex'] = (df['sex'] == 'female').astype(int)          # female=1, male=0
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())

features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']
X = df[features].to_numpy(dtype=float)
y = df['survived'].to_numpy(dtype=int)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y)
print(X_tr.shape, X_te.shape, 'positive rate:', round(float(y.mean()), 3))

In [ ]:
scores = {}   # name -> {'acc':..., 'f1':..., 'auc':...}

def record(name, model):
    """Fit on train, score on test, add a row to the running scoreboard."""
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    scores[name] = dict(acc=accuracy_score(y_te, pred),
                        f1=f1_score(y_te, pred),
                        auc=roc_auc_score(y_te, proba))
    print(f"{name:22s} acc={scores[name]['acc']:.3f}  f1={scores[name]['f1']:.3f}  auc={scores[name]['auc']:.3f}")
    return model

## HW1 - grow, prune, read a tree ([17])

Plot the depth train/test U-curve, prune via `ccp_alpha` chosen by cross-validation, then
`record(...)` the pruned tree. (By-hand Play-Tennis part goes on paper.)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# TODO: sweep max_depth -> U-curve; pick ccp_alpha by CV; visualize with plot_tree.
# tree = record('pruned tree', DecisionTreeClassifier(ccp_alpha=..., random_state=SEED))

## HW2 - bag the tree ([18])

Fit a `RandomForestClassifier(oob_score=True)`, compare OOB vs 5-fold CV, sweep `n_estimators`
(plateau) and `max_features`, read `feature_importances_`, then `record(...)` it.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO: oob vs CV; n_estimators + max_features sweeps; importances.
# rf = record('random forest', RandomForestClassifier(n_estimators=300, oob_score=True,
#                                                      random_state=SEED, n_jobs=-1))

## HW3 - boost the tree ([19]-[20])

Fit gradient boosting with early stopping; then LightGBM with a **monotonic constraint** on a
feature whose effect you expect to be monotone. `record(...)` each.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

# TODO: HistGradientBoosting (early_stopping=True); then lightgbm.LGBMClassifier
#       with monotone_constraints. Compare to your HW2 forest.
# gb = record('hist gradient boosting', HistGradientBoostingClassifier(random_state=SEED))

## The running scoreboard

How does each model compare? On this small, one-dominant-feature dataset the answer is not the
textbook one - see the lecture's "reality check" frame in [18].

In [ ]:
pd.DataFrame(scores).T.round(3)